In [1]:
import random
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, GridSearchCV

import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from torch.utils.data import DataLoader, TensorDataset

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.metrics import classification_report, recall_score, precision_recall_curve, roc_auc_score, confusion_matrix, precision_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import PolynomialFeatures
from scipy.stats import uniform
from sklearn.base import BaseEstimator, TransformerMixin
import xgboost as xgb

/home/hle12/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("/home/hle12/Project/Churn/EDA/post_EDA.csv")

In [3]:
df.head()

,PercChangeMinutes,RevenuePerMinute,CurrentEquipmentDays,MonthlyMinutes,EquipmentAgeRatio,DeviceToServiceAgeRatio,PeakCallsInOut,OffPeakCallsInOut,ServiceIssueRate,CallRatio,...,ReceivedCalls,UnansweredCalls,AgeHH1,TotalRecurringCharge,OutboundCalls,MonthsInService,ServiceAgeInDays,InboundCalls,CustomerCareCalls,Churn_numeric
0,-157.0,0.109091,5.891644,5.393628,0.197160,0.197160,4.077537,3.218876,0.135545,0.000000,...,4.587006,1.987874,62.0,22.0,0.000000,4.127134,7.512618,0.000000,0.000000,1
1,-4.0,1.544545,7.316548,2.397895,0.863871,0.863871,1.791759,0.693147,0.054424,0.000000,...,0.000000,1.308333,40.0,17.0,0.000000,4.077537,7.462215,0.000000,0.000000,1
2,-2.0,4.222222,7.502738,2.197225,1.006108,1.006108,0.832909,1.547563,0.000000,0.262364,...,0.336472,0.000000,26.0,38.0,0.262364,4.110874,7.496097,0.000000,0.000000,0
3,157.0,0.062666,6.129050,7.180070,0.258611,0.258611,6.322026,5.719328,1.448486,1.253342,...,5.304796,4.343805,30.0,75.0,5.917010,4.094345,7.479300,4.997212,1.667707,0
4,0.0,17.140000,6.748760,0.000000,0.535512,0.535512,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,46.0,17.0,0.000000,3.988984,7.372118,0.000000,0.000000,1


In [4]:
df['Churn_numeric'].unique()

array([1, 0])

## Explanation of Results:


### Note: Explore Different Sampling Techniques to Maximise Accuracy

- While for my previous project on domestic violence dataset only explore SMOTE as an option to address SMOTE imbalance, for this dataset I will investigate other resampling methods, including undersampling and hybrid sampling.

- I also draw insights from secondary sources, especially Fujo et al., to hone the methodology of my model.

- How did I choose my threshold? I choose it based on a basic marketing principle: "It is cheaper to keep a customer than to make one."

### Techniques Used:

2. Threshold-Moving
3. Stratified Sampling, Undersampling, Oversampling

Below is the code:

### Split data into test, train and val

In [5]:
transformed = df.copy()

In [6]:
y = transformed['Churn_numeric']
X = transformed.drop(columns = 'Churn_numeric', axis = 1)

In [7]:
y.head()

0    1
1    1
2    0
3    0
4    1
Name: Churn_numeric, dtype: int64

In [8]:
X.head()

,PercChangeMinutes,RevenuePerMinute,CurrentEquipmentDays,MonthlyMinutes,EquipmentAgeRatio,DeviceToServiceAgeRatio,PeakCallsInOut,OffPeakCallsInOut,ServiceIssueRate,CallRatio,...,TotalCalls,ReceivedCalls,UnansweredCalls,AgeHH1,TotalRecurringCharge,OutboundCalls,MonthsInService,ServiceAgeInDays,InboundCalls,CustomerCareCalls
0,-157.0,0.109091,5.891644,5.393628,0.197160,0.197160,4.077537,3.218876,0.135545,0.000000,...,4.587006,4.587006,1.987874,62.0,22.0,0.000000,4.127134,7.512618,0.000000,0.000000
1,-4.0,1.544545,7.316548,2.397895,0.863871,0.863871,1.791759,0.693147,0.054424,0.000000,...,0.000000,0.000000,1.308333,40.0,17.0,0.000000,4.077537,7.462215,0.000000,0.000000
2,-2.0,4.222222,7.502738,2.197225,1.006108,1.006108,0.832909,1.547563,0.000000,0.262364,...,0.530628,0.336472,0.000000,26.0,38.0,0.262364,4.110874,7.496097,0.000000,0.000000
3,157.0,0.062666,6.129050,7.180070,0.258611,0.258611,6.322026,5.719328,1.448486,1.253342,...,6.577305,5.304796,4.343805,30.0,75.0,5.917010,4.094345,7.479300,4.997212,1.667707
4,0.0,17.140000,6.748760,0.000000,0.535512,0.535512,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,46.0,17.0,0.000000,3.988984,7.372118,0.000000,0.000000


## Data Preparation & Transformation

The process for data preparation include:
1. Splitting data
2. Sample training data
3. Convert X and y to numpy
4. Scale X using standard scaler
5. Convert X and y to tensors

We are testing 4 different models: Logistic Regression, XGBoost, Random Forest, Neural Network. These were chosen by process of elimination and based on the progress. For example, we start with Logistic Regression, due to its simiplicity to explain to stakeholders and because the problem is that of classification. Because Logistic Regression is not suffice, we explore models that introduce non-linear relationships. XG boost was chosen because it is a top performer on tabular data and Random Forest was chosen for speed which makes it ideal for larger datasets for large firms. Lastly, Neural Network was chosen because it could capture a more complex relationship, despite requiring more computational resource.

Among these 4 models, only Neural Network needs a validation data set to prevent over-fitting so we will create a separate Train + Val + Test data set for it.

The Train data set will also have 4 different sampling techniques:
(Note that all 4 are combined with stratified sampling)

1. Original / Only Stratified
2. RandomUnderSampler
3. SMOTE
4. Smote + RUS

In [9]:
# Set scaler
scaler = MinMaxScaler()

### Train + Val +Test

In [10]:
# Stratified split into train + temp (validation+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Then split temp into validation and test sets (equal halves here)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [11]:
# Convert training data to numpy for imblearn
X_train_np = X_train.values
y_train_np = y_train.values

In [12]:
X_train_np = scaler.fit_transform(X_train_np)  # Fit on original train

In [13]:
# Because we are using Pytorch, convert the dtype of sets from float64 to tensor.float32.
def to_tensor(X, y):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Add dim for BCELoss
    return X_tensor, y_tensor

In [14]:
# Transform & convert validation and test sets to tensors
X_val = scaler.transform(X_val.values)  
X_test = scaler.transform(X_test.values)

X_val_t, y_val_t = to_tensor(X_val, y_val.values)
X_test_t, y_test_t = to_tensor(X_test, y_test.values)

In [15]:
# Prepare training data 

# Original Training Data
X_train_orig_t, y_train_orig_t = to_tensor(X_train_np, y_train_np)

# Random Under Sampler
undersample = RandomUnderSampler(sampling_strategy='majority', random_state = 42)
X_train_rus_np, y_train_rus_np = undersample.fit_resample(X_train_np, y_train_np)
X_train_rus_np = scaler.fit_transform(X_train_rus_np)
X_train_rus_t, y_train_rus_t = to_tensor(X_train_rus_np, y_train_rus_np)

# Smote
smote  = SMOTE(random_state = 42)
X_train_smote_np, y_train_smote_np = smote.fit_resample(X_train_np, y_train_np)
X_train_smote_np = scaler.fit_transform(X_train_smote_np)
X_train_smote_t, y_train_smote_t = to_tensor(X_train_smote_np, y_train_smote_np)

# SMOTE + Random Under Sampler
pipeline = Pipeline([
    ('smote', smote),
    ('undersample', undersample)
])

X_train_hybrid_np, y_train_hybrid_np = pipeline.fit_resample(X_train_np, y_train_np)
X_train_hybrid_np = scaler.fit_transform(X_train_hybrid_np)
X_train_hybrid_t, y_train_hybrid_t = to_tensor(X_train_hybrid_np, y_train_hybrid_np)

## Train + Test

In [16]:
# Stratified split into train + test
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [17]:
# Because we are using Pytorch, convert the dtype of sets from float64 to tensor.float32.
def to_tensor(X, y):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Add dim for BCELoss
    return X_tensor, y_tensor

In [18]:
# Convert training data to numpy for imblearn
X_train_2_np = X_train_2.values
y_train_2_np = y_train_2.values

# Scale
X_train_2_np = scaler.fit_transform(X_train_2_np)
X_test_2_np =  scaler.transform(X_test_2.values)

# Convert validation and test sets to tensors
X_train_2_t, y_train_2_t = to_tensor(X_train_2_np, y_train_2_np)
X_test_2_t, y_test_2_t = to_tensor(X_test_2_np, y_test_2.values)

In [19]:
# Prepare training data 

# Random Under Sampler
undersample = RandomUnderSampler(sampling_strategy='majority', random_state = 42)
X_train_rus_2_np, y_train_rus_2_np = undersample.fit_resample(X_train_2_np, y_train_2_np)
X_train_rus_2_np = scaler.fit_transform(X_train_rus_2_np)
X_train_rus_2_t, y_train_rus_2_t = to_tensor(X_train_rus_2_np, y_train_rus_2_np)

# Smote
smote  = SMOTE(random_state = 42)
X_train_smote_2_np, y_train_smote_2_np = smote.fit_resample(X_train_2_np, y_train_2_np)
X_train_smote_2_np = scaler.fit_transform(X_train_smote_2_np)
X_train_smote_2_t, y_train_smote_2_t = to_tensor(X_train_smote_2_np, y_train_smote_2_np)

# SMOTE + Random Under Sampler
pipeline = Pipeline([
    ('smote', smote),
    ('undersample', undersample)
])

X_train_hybrid_2_np, y_train_hybrid_2_np = pipeline.fit_resample(X_train_2_np, y_train_2_np)
X_train_hybrid_2_np = scaler.fit_transform(X_train_hybrid_2_np)
X_train_hybrid_2_t, y_train_hybrid_2_t = to_tensor(X_train_hybrid_2_np, y_train_hybrid_2_np)

# Choosing Model

Because the size of the data set is small, simpler models are tested. This include neural network, linear regression and decision trees.

In [20]:
def create_evaluation_metrics(name, y_test, y_pred, y_pred_probs):
    # ROC AUC
    roc_auc = roc_auc_score(y_test, y_pred_probs)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Classification report 
    report = classification_report(y_test, y_pred)

    print(name)
    print("ROC AUC:", roc_auc)
    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", report)

    print("---------------------------")

In [21]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### Logistic Regression

In [22]:
# Custom dummy transformer for skipping steps
class IdentityTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('poly', PolynomialFeatures(include_bias=False)),
    ('smote', IdentityTransformer()),
    ('rus', IdentityTransformer()),
    ('clf', LogisticRegression(max_iter=300, solver='saga', class_weight='balanced'))
])

param_dist = {
    'poly__degree': [1, 2],
    'smote': [SMOTE(random_state=42), IdentityTransformer()],
    'rus': [RandomUnderSampler(random_state=42), IdentityTransformer()],
    'clf__C': uniform(0.01, 1),
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear', 'saga']
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search.fit(X_train_2_np, y_train_2_np.ravel())


print("Best params:", random_search.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


/home/hle12/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/hle12/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/hle12/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/hle12/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/hle12/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/hle12/miniforge3/lib/python3.12/site-packages/sklearn/linea

[CV] END clf__C=0.3845401188473625, clf__penalty=l1, clf__solver=liblinear, poly__degree=1, rus=IdentityTransformer(), smote=SMOTE(random_state=42); total time=17.7min
[CV] END clf__C=0.48537022318211176, clf__penalty=l1, clf__solver=saga, poly__degree=2, rus=IdentityTransformer(), smote=SMOTE(random_state=42); total time= 2.9min
[CV] END clf__C=0.06641157902710025, clf__penalty=l2, clf__solver=saga, poly__degree=2, rus=IdentityTransformer(), smote=IdentityTransformer(); total time=   5.8s
[CV] END clf__C=0.017066305219717408, clf__penalty=l1, clf__solver=liblinear, poly__degree=1, rus=RandomUnderSampler(random_state=42), smote=SMOTE(random_state=42); total time=   2.4s
[CV] END clf__C=0.14949386065204184, clf__penalty=l2, clf__solver=saga, poly__degree=2, rus=RandomUnderSampler(random_state=42), smote=IdentityTransformer(); total time=   6.1s
[CV] END clf__C=0.14949386065204184, clf__penalty=l2, clf__solver=saga, poly__degree=2, rus=RandomUnderSampler(random_state=42), smote=IdentityT

In [29]:
best_model = random_search.best_estimator_
y_probs = best_model.predict_proba(X_test_2_t)[:, 1]
y_pred = (y_probs >= 0.54).astype(int)

create_evaluation_metrics("RandomSearch + SMOTE + Poly + LR", y_test_2_t, y_pred, y_probs)

RandomSearch + SMOTE + Poly + LR
ROC AUC: 0.6115213799574755
Confusion Matrix:
 [[5097 2171]
 [1680 1262]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.75      0.70      0.73      7268
         1.0       0.37      0.43      0.40      2942

    accuracy                           0.62     10210
   macro avg       0.56      0.57      0.56     10210
weighted avg       0.64      0.62      0.63     10210

---------------------------


### Random Forest Classifier

In [52]:
param_grid = {
    'n_estimators': [100, 200],             # Number of trees in the forest
    'max_depth': [5, 10, 15, 20, None],     # Max depth of trees
    'min_samples_leaf': [1, 2, 5, 10],      # Minimum samples at a leaf node
    'min_samples_split': [2, 5, 10],        # Minimum samples to split a node
    'max_features': ['sqrt', 'log2'],       # Features to consider at each split
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train_2_t, y_train_2_t.ravel())

print(grid_search.best_params_)     # Then access best params
best_rf = grid_search.best_estimator_  # Then access best estimator


{'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 200}


In [30]:
def create_RFC(name, X_train, y_train, weight=None, threshold=0.5):
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        max_features='sqrt',
        min_samples_leaf=10,
        min_samples_split=2,
        random_state=42,
        class_weight=weight
    )
    model.fit(X_train, y_train) 

    y_probs = model.predict_proba(X_test_2_t)[:, 1]

    y_pred = (y_probs >= threshold).astype(int)

    print(name)
    create_evaluation_metrics(model, y_test_2_t, y_pred, y_probs)

    return model

In [31]:
# Stratified / Original
create_RFC('Stratified', X_train_2_t, y_train_2_t.ravel(), weight='balanced', threshold = 0.53)

Stratified
RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_leaf=10, n_estimators=200, random_state=42)
ROC AUC: 0.6495248721662282
Confusion Matrix:
 [[5596 1672]
 [1702 1240]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.77      0.77      0.77      7268
         1.0       0.43      0.42      0.42      2942

    accuracy                           0.67     10210
   macro avg       0.60      0.60      0.60     10210
weighted avg       0.67      0.67      0.67     10210

---------------------------


RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_leaf=10, n_estimators=200, random_state=42)

In [32]:
# RandomUnderSampler
create_RFC('RandomUnderSampler', X_train_rus_2_t, y_train_rus_2_t.ravel(), threshold = 0.58)

RandomUnderSampler
RandomForestClassifier(max_depth=15, min_samples_leaf=10, n_estimators=200,
                       random_state=42)
ROC AUC: 0.6326732064829222
Confusion Matrix:
 [[3940 3328]
 [1045 1897]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.79      0.54      0.64      7268
         1.0       0.36      0.64      0.46      2942

    accuracy                           0.57     10210
   macro avg       0.58      0.59      0.55     10210
weighted avg       0.67      0.57      0.59     10210

---------------------------


RandomForestClassifier(max_depth=15, min_samples_leaf=10, n_estimators=200,
                       random_state=42)

In [35]:
# SMOTE
create_RFC('SMOTE', X_train_smote_2_t, y_train_smote_2_t.ravel(), threshold = 0.50)

SMOTE
RandomForestClassifier(max_depth=15, min_samples_leaf=10, n_estimators=200,
                       random_state=42)
ROC AUC: 0.6391284518485622
Confusion Matrix:
 [[5098 2170]
 [1524 1418]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.77      0.70      0.73      7268
         1.0       0.40      0.48      0.43      2942

    accuracy                           0.64     10210
   macro avg       0.58      0.59      0.58     10210
weighted avg       0.66      0.64      0.65     10210

---------------------------


RandomForestClassifier(max_depth=15, min_samples_leaf=10, n_estimators=200,
                       random_state=42)

In [34]:
# SMOTE + RUS
create_RFC('SMOTE + RUS', X_train_hybrid_2_t, y_train_hybrid_2_t.ravel(), threshold = 0.55)

SMOTE + RUS
RandomForestClassifier(max_depth=15, min_samples_leaf=10, n_estimators=200,
                       random_state=42)
ROC AUC: 0.6385883829247678
Confusion Matrix:
 [[6170 1098]
 [2127  815]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.74      0.85      0.79      7268
         1.0       0.43      0.28      0.34      2942

    accuracy                           0.68     10210
   macro avg       0.58      0.56      0.56     10210
weighted avg       0.65      0.68      0.66     10210

---------------------------


RandomForestClassifier(max_depth=15, min_samples_leaf=10, n_estimators=200,
                       random_state=42)

### XGBoost

In [24]:
def objective(trial):
    param = {
        'verbosity': 0,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'use_label_encoder': False,
        'tree_method': 'hist',
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 3)
    }
    
    model = xgb.XGBClassifier(**param, n_estimators=500, random_state=42)
    model.fit(X_train_orig_t, y_train_orig_t)
    
    preds = model.predict_proba(X_val_t)[:, 1]
    auc = roc_auc_score(y_val_t, preds)
    
    return auc

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best trial:")
trial = study.best_trial
print(f"  AUC: {trial.value:.4f}")
print("  Params:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


[I 2025-10-15 21:22:54,987] A new study created in memory with name: no-name-a38ab34a-d028-4a0a-acbe-9b59609dda40
[I 2025-10-15 21:22:56,040] Trial 0 finished with value: 0.6534651585393184 and parameters: {'max_depth': 4, 'learning_rate': 0.013612434250023375, 'subsample': 0.7980018178069564, 'colsample_bytree': 0.733761020806961, 'min_child_weight': 5, 'gamma': 2.9807090602928836, 'scale_pos_weight': 2.328762960559365}. Best is trial 0 with value: 0.6534651585393184.
[I 2025-10-15 21:22:56,753] Trial 1 finished with value: 0.6516814532437246 and parameters: {'max_depth': 8, 'learning_rate': 0.10445192296024573, 'subsample': 0.739174898410369, 'colsample_bytree': 0.6032989393181005, 'min_child_weight': 6, 'gamma': 4.7209164487886826, 'scale_pos_weight': 1.1303516151255706}. Best is trial 0 with value: 0.6534651585393184.
[I 2025-10-15 21:22:58,067] Trial 2 finished with value: 0.6563449961033476 and parameters: {'max_depth': 6, 'learning_rate': 0.027431356799654097, 'subsample': 0.631

Best trial:
  AUC: 0.6586
  Params:
    max_depth: 6
    learning_rate: 0.01339673112907882
    subsample: 0.6462587959957571
    colsample_bytree: 0.7271651088043469
    min_child_weight: 10
    gamma: 4.2664072657631555
    scale_pos_weight: 1.8708571562978937


In [38]:
create_XGBoost('Original', X_train_2_t, y_train_2_t.ravel(), scale_pos_weight= 1.8708571562978937, threshold = 0.48)

/home/hle12/miniforge3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [21:26:25] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Original
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7271651088043469, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.2664072657631555, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01339673112907882,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)
ROC AUC: 0.6558071252432368
Confusion Matrix:
 [[5387 1881]
 [1582 1360]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7271651088043469, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.2664072657631555, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01339673112907882,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [33]:
def create_XGBoost(name, X_train, y_train, scale_pos_weight = 1, threshold = 0.5):
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate= 0.01339673112907882,
        subsample= 0.6462587959957571,
        colsample_bytree= 0.7271651088043469,
        scale_pos_weight= scale_pos_weight, 
        random_state=42,
        use_label_encoder=False,
        gamma =  4.2664072657631555,
        min_child_weight = 10,
        eval_metric='logloss'
    )
    model.fit(X_train, y_train)

    y_probs = model.predict_proba(X_test_2_t)[:, 1]
    y_pred = (y_probs >= threshold).astype(int)

    print(name)
    create_evaluation_metrics(model, y_test_2_t, y_pred, y_probs)

    return model

In [46]:
create_XGBoost('RUS', X_train_rus_2_t, y_train_rus_2_t.ravel(), threshold = 0.63)

/home/hle12/miniforge3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [21:27:40] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


RUS
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7271651088043469, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.2664072657631555, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01339673112907882,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)
ROC AUC: 0.630919362116307
Confusion Matrix:
 [[4584 2684]
 [1303 1639]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.78    

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7271651088043469, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.2664072657631555, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01339673112907882,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [47]:
create_XGBoost('SMOTE', X_train_smote_2_t, y_train_smote_2_t.ravel())

/home/hle12/miniforge3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [21:28:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


SMOTE
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7271651088043469, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.2664072657631555, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01339673112907882,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)
ROC AUC: 0.6303289014133829
Confusion Matrix:
 [[5182 2086]
 [1634 1308]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.76 

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7271651088043469, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.2664072657631555, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01339673112907882,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [48]:
create_XGBoost('hybrid', X_train_hybrid_2_t, y_train_hybrid_2_t.ravel())

/home/hle12/miniforge3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [21:28:17] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


hybrid
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7271651088043469, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.2664072657631555, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01339673112907882,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)
ROC AUC: 0.6285483529113775
Confusion Matrix:
 [[5174 2094]
 [1628 1314]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.76

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7271651088043469, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=4.2664072657631555, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01339673112907882,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=10, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

### Neural Networks

In [23]:
def objective(trial):
    set_seed(42)

    # Hyperparameters 
    hidden_size1 = trial.suggest_int("hidden_size1", 64, 512, step=32)
    hidden_size2 = trial.suggest_int("hidden_size2", 8, 256, step=8)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    activation_name = trial.suggest_categorical("activation", ["ReLU", "Tanh", "LeakyReLU"])

    # Activation function
    activation = {
        "ReLU": nn.ReLU(),
        "Tanh": nn.Tanh(),
        "LeakyReLU": nn.LeakyReLU()
    }[activation_name]

    # Model
    model = nn.Sequential(
        nn.Linear(21, hidden_size1),
        activation,
        nn.Linear(hidden_size1, hidden_size2),
        activation,
        nn.Linear(hidden_size2, 1),
        nn.Sigmoid()
    )

    model.to('cpu')  # or 'cuda' if using GPU
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    patience = 10
    best_f1 = 0
    epochs_no_improve = 0

    for epoch in range(100):
        # Train
        model.train()
        y_pred = model(X_train_rus_t).squeeze(1)
        loss = loss_fn(y_pred, y_train_rus_t.squeeze(1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Evaluate on validation set
        model.eval()
        with torch.no_grad():
            val_probs = model(X_val_t).squeeze(1).numpy()
            val_preds = (val_probs >= 0.5).astype(int)
            y_val_np = y_val_t.squeeze(1).numpy()
            f1 = f1_score(y_val_np, val_preds, zero_division=0)

        # Early stopping on F1
        if f1 > best_f1:
            best_f1 = f1
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    return best_f1

[CV] END clf__C=0.3845401188473625, clf__penalty=l1, clf__solver=liblinear, poly__degree=1, rus=IdentityTransformer(), smote=SMOTE(random_state=42); total time=24.5min
[CV] END clf__C=0.3845401188473625, clf__penalty=l1, clf__solver=liblinear, poly__degree=1, rus=IdentityTransformer(), smote=SMOTE(random_state=42); total time=25.1min
[CV] END clf__C=0.3845401188473625, clf__penalty=l1, clf__solver=liblinear, poly__degree=1, rus=IdentityTransformer(), smote=SMOTE(random_state=42); total time=24.3min
[CV] END clf__C=0.606850157946487, clf__penalty=l2, clf__solver=liblinear, poly__degree=1, rus=RandomUnderSampler(random_state=42), smote=SMOTE(random_state=42); total time=   3.0s
[CV] END clf__C=0.017066305219717408, clf__penalty=l1, clf__solver=liblinear, poly__degree=1, rus=RandomUnderSampler(random_state=42), smote=SMOTE(random_state=42); total time=  13.4s
[CV] END clf__C=0.14949386065204184, clf__penalty=l2, clf__solver=saga, poly__degree=2, rus=RandomUnderSampler(random_state=42), sm

In [24]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)
print("Best hyperparameters:", study.best_params)

[I 2025-10-16 20:16:48,317] A new study created in memory with name: no-name-79ab842a-a127-4735-ba11-47f10ceb02e8
[I 2025-10-16 20:16:52,876] Trial 0 finished with value: 0.44754173686628884 and parameters: {'hidden_size1': 416, 'hidden_size2': 128, 'lr': 0.007761561711965477, 'activation': 'LeakyReLU'}. Best is trial 0 with value: 0.44754173686628884.
[I 2025-10-16 20:16:53,268] Trial 1 finished with value: 0.4495631453534551 and parameters: {'hidden_size1': 160, 'hidden_size2': 104, 'lr': 0.0005178935584870791, 'activation': 'Tanh'}. Best is trial 1 with value: 0.4495631453534551.
[I 2025-10-16 20:16:53,697] Trial 2 finished with value: 0.4556586270871985 and parameters: {'hidden_size1': 160, 'hidden_size2': 104, 'lr': 0.007781495749155925, 'activation': 'Tanh'}. Best is trial 2 with value: 0.4556586270871985.
[I 2025-10-16 20:16:54,374] Trial 3 finished with value: 0.45652173913043476 and parameters: {'hidden_size1': 256, 'hidden_size2': 112, 'lr': 0.00041270826662052716, 'activatio

Best hyperparameters: {'hidden_size1': 224, 'hidden_size2': 144, 'lr': 0.005290717457324993, 'activation': 'LeakyReLU'}


In [25]:
def build_model():
    input_dim = X_train.shape[1]
    
    model = nn.Sequential(
    nn.Linear(input_dim, 224),
    nn.LeakyReLU(),
    nn.Linear(224, 144),
    nn.LeakyReLU(),
    nn.Linear(144, 1),
    nn.Sigmoid()
    )
    
    return model

In [26]:
def train_model(model, train_loader, epochs=100, patience=10):
    set_seed(42)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr= 0.005290717457324993)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-4)

    best_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        scheduler.step(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

In [27]:
def test_model(name, model, X_test_t, y_test_t, threshold=0.65):
    set_seed(42)
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t).squeeze(1)
        probs = torch.sigmoid(logits)  # convert logits to probabilities
        y_pred = (probs > threshold).int()

        y_true = y_test_t.squeeze(1).cpu().numpy()
        y_pred_np = y_pred.cpu().numpy()
        probs_np = probs.cpu().numpy()

    create_evaluation_metrics(name, y_true, y_pred_np, probs_np)

In [28]:
input_dim = X_train_orig_t.shape[1]

In [29]:
orig_train_dataset = TensorDataset(X_train_orig_t, y_train_orig_t)
orig_train_loader = DataLoader(orig_train_dataset, batch_size=32, shuffle=True)

In [30]:
NN_orig = build_model()

train_model(NN_orig, orig_train_loader, epochs=100)

Epoch [1/100], Loss: 0.5936
Epoch [10/100], Loss: 0.5800
Epoch [20/100], Loss: 0.5785
Epoch [30/100], Loss: 0.5773
Epoch [40/100], Loss: 0.5760
Epoch [50/100], Loss: 0.5747
Epoch [60/100], Loss: 0.5739
Epoch [70/100], Loss: 0.5735
Epoch [80/100], Loss: 0.5737
Epoch [90/100], Loss: 0.5698
Epoch [100/100], Loss: 0.5682


In [37]:
test_model("Stratified", NN_orig, X_test_t, y_test_t, threshold = 0.59)

Stratified
ROC AUC: 0.6298339161787588
Confusion Matrix:
 [[2788  846]
 [ 887  584]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.76      0.77      0.76      3634
         1.0       0.41      0.40      0.40      1471

    accuracy                           0.66      5105
   macro avg       0.58      0.58      0.58      5105
weighted avg       0.66      0.66      0.66      5105

---------------------------


In [38]:
rus_train_dataset = TensorDataset(X_train_rus_t, y_train_rus_t)
rus_train_loader = DataLoader(rus_train_dataset, batch_size=32, shuffle=True)

In [39]:
NN_rus = build_model()

train_model(NN_rus, rus_train_loader, epochs=100)

Epoch [1/100], Loss: 0.6860
Epoch [10/100], Loss: 0.6668
Epoch [20/100], Loss: 0.6646
Epoch [30/100], Loss: 0.6625
Epoch [40/100], Loss: 0.6604
Epoch [50/100], Loss: 0.6555
Epoch [60/100], Loss: 0.6536
Epoch [70/100], Loss: 0.6525
Epoch [80/100], Loss: 0.6500
Epoch [90/100], Loss: 0.6475
Epoch [100/100], Loss: 0.6453


In [40]:
test_model("RandomUnderSampling", NN_rus, X_test_t, y_test_t)

RandomUnderSampling
ROC AUC: 0.6243919781712635
Confusion Matrix:
 [[2855  779]
 [ 945  526]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.75      0.79      0.77      3634
         1.0       0.40      0.36      0.38      1471

    accuracy                           0.66      5105
   macro avg       0.58      0.57      0.57      5105
weighted avg       0.65      0.66      0.66      5105

---------------------------


In [ ]:
smote_train_dataset = TensorDataset(X_train_smote_t, y_train_smote_t)
smote_train_loader = DataLoader(smote_train_dataset, batch_size=32, shuffle=True)

In [42]:
NN_smote = build_model()

train_model(NN_smote, smote_train_loader, epochs=100)

Epoch [1/100], Loss: 0.6775
Epoch [10/100], Loss: 0.6604
Epoch [20/100], Loss: 0.6563
Epoch [30/100], Loss: 0.6531
Epoch [40/100], Loss: 0.6501
Epoch [50/100], Loss: 0.6473
Epoch [60/100], Loss: 0.6456
Epoch [70/100], Loss: 0.6431
Epoch [80/100], Loss: 0.6413
Epoch [90/100], Loss: 0.6386
Epoch [100/100], Loss: 0.6368


In [43]:
test_model("Smote", NN_smote, X_test_t, y_test_t)

Smote
ROC AUC: 0.6230983007751776
Confusion Matrix:
 [[2956  678]
 [1004  467]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.75      0.81      0.78      3634
         1.0       0.41      0.32      0.36      1471

    accuracy                           0.67      5105
   macro avg       0.58      0.57      0.57      5105
weighted avg       0.65      0.67      0.66      5105

---------------------------


In [ ]:
hybrid_train_dataset = TensorDataset(X_train_hybrid_t, y_train_hybrid_t)
hybrid_train_loader = DataLoader(hybrid_train_dataset, batch_size=32, shuffle=True)

In [45]:
NN_hybrid = build_model()

train_model(NN_hybrid, hybrid_train_loader, epochs=100)

Epoch [1/100], Loss: 0.6767
Epoch [10/100], Loss: 0.6623
Epoch [20/100], Loss: 0.6577
Epoch [30/100], Loss: 0.6547
Epoch [40/100], Loss: 0.6524
Epoch [50/100], Loss: 0.6495
Epoch [60/100], Loss: 0.6482
Epoch [70/100], Loss: 0.6456
Epoch [80/100], Loss: 0.6437
Epoch [90/100], Loss: 0.6404
Epoch [100/100], Loss: 0.6387


In [46]:
test_model("Hybrid", NN_hybrid, X_test_t, y_test_t)

Hybrid
ROC AUC: 0.6237717500739859
Confusion Matrix:
 [[2983  651]
 [1020  451]]
Classification Report:
               precision    recall  f1-score   support

         0.0       0.75      0.82      0.78      3634
         1.0       0.41      0.31      0.35      1471

    accuracy                           0.67      5105
   macro avg       0.58      0.56      0.57      5105
weighted avg       0.65      0.67      0.66      5105

---------------------------


## Best model:

<b> RandomForestClassifier with Stratified Sampling and Class Weights </b>

Key Metrics:

- ROC AUC: 🟢 0.6495 (highest among all models)
- Recall (class 1 - churn): 0.42
- Precision (class 1 - churn): 0.43
- F1-score (class 1): 0.42
- Accuracy: 0.67

Why this model?
- Highest ROC AUC: It best distinguishes between churners and non-churners across thresholds.
- Balanced accuracy and F1: It maintains a reasonable balance between detecting churners and avoiding false alarms.
- Good performance on imbalanced data: The class_weight='balanced' parameter compensates for class imbalance without needing resampling tricks like SMOTE or RUS, which can sometimes introduce noise.